In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm


from paddy_10_data_loader import load_train_val_test_data
from shufflenet_v2 import ShuffleNetV2
from helper_utils import show_sample_images, training_loop, plot_training_history, visualize_predictions, plot_confusion_matrix

import kd_utils

from fit_net_wrapper import FitNetWrapper

In [2]:
student_train_loader, student_val_loader, student_test_loader = load_train_val_test_data(batch_size=32)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
densenet_201_teacher = timm.create_model("densenet201", pretrained=False, num_classes=10)
densenet_201_teacher.load_state_dict(torch.load("densenet_201_teacher.pth"))

<All keys matched successfully>

# Train with FitNet

In [5]:
hint_epochs = 15
train_epochs = 15

In [6]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.50,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_50.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2333
Epoch 2/15 - Hint Loss: 1.7858
Epoch 3/15 - Hint Loss: 1.6274
Epoch 4/15 - Hint Loss: 1.5406
Epoch 5/15 - Hint Loss: 1.4884
Epoch 6/15 - Hint Loss: 1.4529
Epoch 7/15 - Hint Loss: 1.4291
Epoch 8/15 - Hint Loss: 1.4084
Epoch 9/15 - Hint Loss: 1.3934
Epoch 10/15 - Hint Loss: 1.3819
Epoch 11/15 - Hint Loss: 1.3733
Epoch 12/15 - Hint Loss: 1.3660
Epoch 13/15 - Hint Loss: 1.3605
Epoch 14/15 - Hint Loss: 1.3601
Epoch 15/15 - Hint Loss: 1.3565


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 56.9647%, Train Hard Loss: 1.4935, Train Soft Loss: 0.6082, Train Distill Loss: 5.6122 | Val Hard Loss: 1.7703, Val Soft Loss: 0.5151, Val Distill Loss: 5.0058, Val Acc: 57.7938%
Epoch 2/15 - Train Acc: 76.7994%, Train Hard Loss: 0.8632, Train Soft Loss: 0.3200, Train Distill Loss: 2.9913 | Val Hard Loss: 0.8236, Val Soft Loss: 0.2300, Val Distill Loss: 2.2515, Val Acc: 80.0360%
Epoch 3/15 - Train Acc: 84.5229%, Train Hard Loss: 0.5502, Train Soft Loss: 0.2225, Train Distill Loss: 2.0549 | Val Hard Loss: 0.6504, Val Soft Loss: 0.1887, Val Distill Loss: 1.8347, Val Acc: 84.1127%
Epoch 4/15 - Train Acc: 90.8039%, Train Hard Loss: 0.3210, Train Soft Loss: 0.1576, Train Distill Loss: 1.4214 | Val Hard Loss: 0.5884, Val Soft Loss: 0.1642, Val Distill Loss: 1.6078, Val Acc: 86.0911%
Epoch 5/15 - Train Acc: 94.0496%, Train Hard Loss: 0.1942, Train Soft Loss: 0.1245, Train Distill Loss: 1.0929 | Val Hard Loss: 0.5344, Val Soft Loss: 0.1689, Val Distill Loss: 1.6186, Val

0.9361804222648752

In [7]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.55,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_55.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2441
Epoch 2/15 - Hint Loss: 1.7894
Epoch 3/15 - Hint Loss: 1.6246
Epoch 4/15 - Hint Loss: 1.5400
Epoch 5/15 - Hint Loss: 1.4878
Epoch 6/15 - Hint Loss: 1.4524
Epoch 7/15 - Hint Loss: 1.4277
Epoch 8/15 - Hint Loss: 1.4086
Epoch 9/15 - Hint Loss: 1.3935
Epoch 10/15 - Hint Loss: 1.3819
Epoch 11/15 - Hint Loss: 1.3734
Epoch 12/15 - Hint Loss: 1.3655
Epoch 13/15 - Hint Loss: 1.3594
Epoch 14/15 - Hint Loss: 1.3547
Epoch 15/15 - Hint Loss: 1.3848


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 56.3937%, Train Hard Loss: 1.5457, Train Soft Loss: 0.6343, Train Distill Loss: 5.4170 | Val Hard Loss: 1.5419, Val Soft Loss: 0.4547, Val Distill Loss: 4.4085, Val Acc: 63.3094%
Epoch 2/15 - Train Acc: 75.7626%, Train Hard Loss: 0.9140, Train Soft Loss: 0.3445, Train Distill Loss: 2.9828 | Val Hard Loss: 0.9163, Val Soft Loss: 0.2553, Val Distill Loss: 2.5005, Val Acc: 77.0384%
Epoch 3/15 - Train Acc: 84.3276%, Train Hard Loss: 0.5674, Train Soft Loss: 0.2343, Train Distill Loss: 1.9993 | Val Hard Loss: 0.6556, Val Soft Loss: 0.1999, Val Distill Loss: 1.9270, Val Acc: 83.6331%
Epoch 4/15 - Train Acc: 88.7303%, Train Hard Loss: 0.3872, Train Soft Loss: 0.1811, Train Distill Loss: 1.5171 | Val Hard Loss: 0.5279, Val Soft Loss: 0.1586, Val Distill Loss: 1.5328, Val Acc: 87.8897%
Epoch 5/15 - Train Acc: 92.3366%, Train Hard Loss: 0.2362, Train Soft Loss: 0.1374, Train Distill Loss: 1.1193 | Val Hard Loss: 0.4013, Val Soft Loss: 0.1194, Val Distill Loss: 1.1555, Val

0.9333013435700576

In [8]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.60,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_60.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2361
Epoch 2/15 - Hint Loss: 1.7789
Epoch 3/15 - Hint Loss: 1.6213
Epoch 4/15 - Hint Loss: 1.5411
Epoch 5/15 - Hint Loss: 1.4890
Epoch 6/15 - Hint Loss: 1.4531
Epoch 7/15 - Hint Loss: 1.4267
Epoch 8/15 - Hint Loss: 1.4107
Epoch 9/15 - Hint Loss: 1.3929
Epoch 10/15 - Hint Loss: 1.3813
Epoch 11/15 - Hint Loss: 1.3730
Epoch 12/15 - Hint Loss: 1.3653
Epoch 13/15 - Hint Loss: 1.3597
Epoch 14/15 - Hint Loss: 1.3551
Epoch 15/15 - Hint Loss: 1.3512


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 55.6724%, Train Hard Loss: 1.5226, Train Soft Loss: 0.6404, Train Distill Loss: 5.0119 | Val Hard Loss: 1.5745, Val Soft Loss: 0.5045, Val Distill Loss: 4.8232, Val Acc: 58.0935%
Epoch 2/15 - Train Acc: 75.6123%, Train Hard Loss: 0.8905, Train Soft Loss: 0.3370, Train Distill Loss: 2.6913 | Val Hard Loss: 0.8295, Val Soft Loss: 0.2241, Val Distill Loss: 2.2072, Val Acc: 77.2782%
Epoch 3/15 - Train Acc: 83.6364%, Train Hard Loss: 0.5748, Train Soft Loss: 0.2317, Train Distill Loss: 1.8278 | Val Hard Loss: 1.0345, Val Soft Loss: 0.2944, Val Distill Loss: 2.8728, Val Acc: 77.3381%
Epoch 4/15 - Train Acc: 89.6469%, Train Hard Loss: 0.3492, Train Soft Loss: 0.1651, Train Distill Loss: 1.2664 | Val Hard Loss: 0.5323, Val Soft Loss: 0.1518, Val Distill Loss: 1.4803, Val Acc: 85.9113%
Epoch 5/15 - Train Acc: 92.8174%, Train Hard Loss: 0.2369, Train Soft Loss: 0.1351, Train Distill Loss: 1.0068 | Val Hard Loss: 0.4287, Val Soft Loss: 0.1255, Val Distill Loss: 1.2187, Val

0.9347408829174664

In [9]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.65,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_65.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2330
Epoch 2/15 - Hint Loss: 1.7739
Epoch 3/15 - Hint Loss: 1.6152
Epoch 4/15 - Hint Loss: 1.5321
Epoch 5/15 - Hint Loss: 1.4834
Epoch 6/15 - Hint Loss: 1.4482
Epoch 7/15 - Hint Loss: 1.4260
Epoch 8/15 - Hint Loss: 1.4062
Epoch 9/15 - Hint Loss: 1.3911
Epoch 10/15 - Hint Loss: 1.3796
Epoch 11/15 - Hint Loss: 1.3707
Epoch 12/15 - Hint Loss: 1.3640
Epoch 13/15 - Hint Loss: 1.3592
Epoch 14/15 - Hint Loss: 1.3545
Epoch 15/15 - Hint Loss: 1.3501


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.4005%, Train Hard Loss: 1.4354, Train Soft Loss: 0.6271, Train Distill Loss: 4.4449 | Val Hard Loss: 2.0683, Val Soft Loss: 0.5789, Val Distill Loss: 5.6655, Val Acc: 68.3453%
Epoch 2/15 - Train Acc: 76.6191%, Train Hard Loss: 0.8608, Train Soft Loss: 0.3329, Train Distill Loss: 2.4238 | Val Hard Loss: 0.7647, Val Soft Loss: 0.2112, Val Distill Loss: 2.0719, Val Acc: 79.9760%
Epoch 3/15 - Train Acc: 84.4929%, Train Hard Loss: 0.5419, Train Soft Loss: 0.2239, Train Distill Loss: 1.6059 | Val Hard Loss: 0.7640, Val Soft Loss: 0.2136, Val Distill Loss: 2.0905, Val Acc: 82.1343%
Epoch 4/15 - Train Acc: 90.3231%, Train Hard Loss: 0.3262, Train Soft Loss: 0.1634, Train Distill Loss: 1.1273 | Val Hard Loss: 0.6618, Val Soft Loss: 0.1967, Val Distill Loss: 1.9045, Val Acc: 82.8537%
Epoch 5/15 - Train Acc: 93.1180%, Train Hard Loss: 0.2129, Train Soft Loss: 0.1330, Train Distill Loss: 0.8834 | Val Hard Loss: 0.3238, Val Soft Loss: 0.1168, Val Distill Loss: 1.0960, Val

0.9428982725527831

In [10]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.70,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_70.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2482
Epoch 2/15 - Hint Loss: 1.7845
Epoch 3/15 - Hint Loss: 1.6210
Epoch 4/15 - Hint Loss: 1.5333
Epoch 5/15 - Hint Loss: 1.4809
Epoch 6/15 - Hint Loss: 1.4464
Epoch 7/15 - Hint Loss: 1.4213
Epoch 8/15 - Hint Loss: 1.4026
Epoch 9/15 - Hint Loss: 1.3883
Epoch 10/15 - Hint Loss: 1.3784
Epoch 11/15 - Hint Loss: 1.3698
Epoch 12/15 - Hint Loss: 1.3632
Epoch 13/15 - Hint Loss: 1.3578
Epoch 14/15 - Hint Loss: 1.3546
Epoch 15/15 - Hint Loss: 1.3519


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.0849%, Train Hard Loss: 1.4168, Train Soft Loss: 0.6448, Train Distill Loss: 4.0867 | Val Hard Loss: 1.1959, Val Soft Loss: 0.3676, Val Distill Loss: 3.5386, Val Acc: 68.9448%
Epoch 2/15 - Train Acc: 77.1450%, Train Hard Loss: 0.8125, Train Soft Loss: 0.3314, Train Distill Loss: 2.1596 | Val Hard Loss: 0.8839, Val Soft Loss: 0.2435, Val Distill Loss: 2.3901, Val Acc: 77.1583%
Epoch 3/15 - Train Acc: 84.8084%, Train Hard Loss: 0.5377, Train Soft Loss: 0.2325, Train Distill Loss: 1.4926 | Val Hard Loss: 0.8805, Val Soft Loss: 0.2433, Val Distill Loss: 2.3864, Val Acc: 78.2974%
Epoch 4/15 - Train Acc: 89.4666%, Train Hard Loss: 0.3482, Train Soft Loss: 0.1725, Train Distill Loss: 1.0719 | Val Hard Loss: 0.6906, Val Soft Loss: 0.1899, Val Distill Loss: 1.8649, Val Acc: 82.8537%
Epoch 5/15 - Train Acc: 93.3283%, Train Hard Loss: 0.2069, Train Soft Loss: 0.1347, Train Distill Loss: 0.7912 | Val Hard Loss: 0.4298, Val Soft Loss: 0.1439, Val Distill Loss: 1.3664, Val

0.9404990403071017

In [11]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.75,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_75.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2448
Epoch 2/15 - Hint Loss: 1.7928
Epoch 3/15 - Hint Loss: 1.6264
Epoch 4/15 - Hint Loss: 1.5441
Epoch 5/15 - Hint Loss: 1.4918
Epoch 6/15 - Hint Loss: 1.4566
Epoch 7/15 - Hint Loss: 1.4312
Epoch 8/15 - Hint Loss: 1.4104
Epoch 9/15 - Hint Loss: 1.3950
Epoch 10/15 - Hint Loss: 1.3832
Epoch 11/15 - Hint Loss: 1.3738
Epoch 12/15 - Hint Loss: 1.3674
Epoch 13/15 - Hint Loss: 1.3602
Epoch 14/15 - Hint Loss: 1.3563
Epoch 15/15 - Hint Loss: 1.3526


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 56.1533%, Train Hard Loss: 1.4036, Train Soft Loss: 0.6488, Train Distill Loss: 3.6478 | Val Hard Loss: 1.0841, Val Soft Loss: 0.3658, Val Distill Loss: 3.4682, Val Acc: 69.3046%
Epoch 2/15 - Train Acc: 77.1300%, Train Hard Loss: 0.8383, Train Soft Loss: 0.3475, Train Distill Loss: 2.0188 | Val Hard Loss: 1.4932, Val Soft Loss: 0.3885, Val Distill Loss: 3.8543, Val Acc: 64.5683%
Epoch 3/15 - Train Acc: 83.6965%, Train Hard Loss: 0.5715, Train Soft Loss: 0.2511, Train Distill Loss: 1.4332 | Val Hard Loss: 1.0312, Val Soft Loss: 0.2846, Val Distill Loss: 2.7924, Val Acc: 75.6595%
Epoch 4/15 - Train Acc: 88.4448%, Train Hard Loss: 0.3930, Train Soft Loss: 0.1925, Train Distill Loss: 1.0646 | Val Hard Loss: 0.6614, Val Soft Loss: 0.1966, Val Distill Loss: 1.9038, Val Acc: 84.1127%
Epoch 5/15 - Train Acc: 92.3065%, Train Hard Loss: 0.2462, Train Soft Loss: 0.1473, Train Distill Loss: 0.7740 | Val Hard Loss: 0.4592, Val Soft Loss: 0.1497, Val Distill Loss: 1.4276, Val

0.9361804222648752

In [12]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.80,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_80.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2397
Epoch 2/15 - Hint Loss: 1.7807
Epoch 3/15 - Hint Loss: 1.6186
Epoch 4/15 - Hint Loss: 1.5350
Epoch 5/15 - Hint Loss: 1.4832
Epoch 6/15 - Hint Loss: 1.4474
Epoch 7/15 - Hint Loss: 1.4225
Epoch 8/15 - Hint Loss: 1.4091
Epoch 9/15 - Hint Loss: 1.3914
Epoch 10/15 - Hint Loss: 1.3803
Epoch 11/15 - Hint Loss: 1.3720
Epoch 12/15 - Hint Loss: 1.3654
Epoch 13/15 - Hint Loss: 1.3599
Epoch 14/15 - Hint Loss: 1.3573
Epoch 15/15 - Hint Loss: 1.3518


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 56.5890%, Train Hard Loss: 1.3446, Train Soft Loss: 0.6585, Train Distill Loss: 3.1830 | Val Hard Loss: 1.4626, Val Soft Loss: 0.4390, Val Distill Loss: 4.2431, Val Acc: 64.6283%
Epoch 2/15 - Train Acc: 76.8745%, Train Hard Loss: 0.8053, Train Soft Loss: 0.3551, Train Distill Loss: 1.7806 | Val Hard Loss: 0.8973, Val Soft Loss: 0.2947, Val Distill Loss: 2.8063, Val Acc: 74.4005%
Epoch 3/15 - Train Acc: 83.9820%, Train Hard Loss: 0.5546, Train Soft Loss: 0.2480, Train Distill Loss: 1.2372 | Val Hard Loss: 0.7145, Val Soft Loss: 0.2402, Val Distill Loss: 2.2789, Val Acc: 79.7362%
Epoch 4/15 - Train Acc: 88.8805%, Train Hard Loss: 0.3565, Train Soft Loss: 0.1923, Train Distill Loss: 0.9006 | Val Hard Loss: 0.6869, Val Soft Loss: 0.2259, Val Distill Loss: 2.1507, Val Acc: 81.8945%
Epoch 5/15 - Train Acc: 92.6221%, Train Hard Loss: 0.2227, Train Soft Loss: 0.1452, Train Distill Loss: 0.6429 | Val Hard Loss: 0.4904, Val Soft Loss: 0.1413, Val Distill Loss: 1.3759, Val

0.9438579654510557

In [13]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.85,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_85.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2146
Epoch 2/15 - Hint Loss: 1.7579
Epoch 3/15 - Hint Loss: 1.6068
Epoch 4/15 - Hint Loss: 1.5285
Epoch 5/15 - Hint Loss: 1.4805
Epoch 6/15 - Hint Loss: 1.4525
Epoch 7/15 - Hint Loss: 1.4254
Epoch 8/15 - Hint Loss: 1.4067
Epoch 9/15 - Hint Loss: 1.3929
Epoch 10/15 - Hint Loss: 1.3817
Epoch 11/15 - Hint Loss: 1.3732
Epoch 12/15 - Hint Loss: 1.3681
Epoch 13/15 - Hint Loss: 1.3617
Epoch 14/15 - Hint Loss: 1.3574
Epoch 15/15 - Hint Loss: 1.3536


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.2652%, Train Hard Loss: 1.3287, Train Soft Loss: 0.6674, Train Distill Loss: 2.7311 | Val Hard Loss: 1.0412, Val Soft Loss: 0.4009, Val Distill Loss: 3.7281, Val Acc: 67.8657%
Epoch 2/15 - Train Acc: 76.5289%, Train Hard Loss: 0.8081, Train Soft Loss: 0.3716, Train Distill Loss: 1.5788 | Val Hard Loss: 0.8153, Val Soft Loss: 0.2679, Val Distill Loss: 2.5506, Val Acc: 74.7602%
Epoch 3/15 - Train Acc: 83.3208%, Train Hard Loss: 0.5647, Train Soft Loss: 0.2685, Train Distill Loss: 1.1245 | Val Hard Loss: 0.8084, Val Soft Loss: 0.2682, Val Distill Loss: 2.5497, Val Acc: 78.7770%
Epoch 4/15 - Train Acc: 87.0924%, Train Hard Loss: 0.4255, Train Soft Loss: 0.2172, Train Distill Loss: 0.8831 | Val Hard Loss: 0.5759, Val Soft Loss: 0.1854, Val Distill Loss: 1.7707, Val Acc: 83.5132%
Epoch 5/15 - Train Acc: 91.9609%, Train Hard Loss: 0.2560, Train Soft Loss: 0.1601, Train Distill Loss: 0.6017 | Val Hard Loss: 0.4615, Val Soft Loss: 0.1645, Val Distill Loss: 1.5465, Val

0.9390595009596929

In [14]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4,
    alpha=0.90,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path="fitnet_shufflenet_v2_15_4_90.pth",
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2351
Epoch 2/15 - Hint Loss: 1.7921
Epoch 3/15 - Hint Loss: 1.6236
Epoch 4/15 - Hint Loss: 1.5379
Epoch 5/15 - Hint Loss: 1.4864
Epoch 6/15 - Hint Loss: 1.4520
Epoch 7/15 - Hint Loss: 1.4263
Epoch 8/15 - Hint Loss: 1.4067
Epoch 9/15 - Hint Loss: 1.3919
Epoch 10/15 - Hint Loss: 1.3800
Epoch 11/15 - Hint Loss: 1.3713
Epoch 12/15 - Hint Loss: 1.3646
Epoch 13/15 - Hint Loss: 1.3605
Epoch 14/15 - Hint Loss: 1.3552
Epoch 15/15 - Hint Loss: 1.3520


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 58.3922%, Train Hard Loss: 1.2568, Train Soft Loss: 0.6780, Train Distill Loss: 2.2159 | Val Hard Loss: 1.0552, Val Soft Loss: 0.3819, Val Distill Loss: 3.5829, Val Acc: 70.9233%
Epoch 2/15 - Train Acc: 75.3419%, Train Hard Loss: 0.7831, Train Soft Loss: 0.3914, Train Distill Loss: 1.3310 | Val Hard Loss: 1.2700, Val Soft Loss: 0.3761, Val Distill Loss: 3.6440, Val Acc: 69.1847%
Epoch 3/15 - Train Acc: 82.6446%, Train Hard Loss: 0.5614, Train Soft Loss: 0.2931, Train Distill Loss: 0.9742 | Val Hard Loss: 0.7265, Val Soft Loss: 0.2344, Val Distill Loss: 2.2387, Val Acc: 79.3765%
Epoch 4/15 - Train Acc: 87.2577%, Train Hard Loss: 0.3948, Train Soft Loss: 0.2244, Train Distill Loss: 0.7143 | Val Hard Loss: 0.6113, Val Soft Loss: 0.1864, Val Distill Loss: 1.7972, Val Acc: 83.8130%
Epoch 5/15 - Train Acc: 91.1645%, Train Hard Loss: 0.2791, Train Soft Loss: 0.1779, Train Distill Loss: 0.5357 | Val Hard Loss: 0.7452, Val Soft Loss: 0.2204, Val Distill Loss: 2.1356, Val

0.9304222648752399